# Experiment: Baby Cry Model Bakeoff

This notebook turns the current baby-cry baseline into a **model comparison and fine-tuning lab**.

It is designed for the real repository layout under `training/`, so it will automatically reuse the already-downloaded public Donate-a-Cry data when present.


## Why this notebook exists

The production app is still wired to a **mock classifier**, which is fine for product iteration but not for model selection.

This notebook gives you a practical research workflow:

1. Reuse the current Donate-a-Cry dataset and the same label mapping used by the app.
2. Keep the existing **group-based split** logic so we do not leak clips from the same source group across train/val/test.
3. Compare a small set of strong, realistic baselines for tiny and imbalanced audio datasets.
4. Keep a path open for **fine-tuning** the strongest candidate after the fast bakeoff.

The core idea is to avoid overcommitting too early to one architecture when the dataset is still small and noisy.


## Model strategy

This notebook uses two evaluation layers.

### Layer 1: fast model bakeoff

These are cheap enough to compare on a laptop:

- `Mel CNN baseline`: the current spectrogram baseline from `train_baseline.py`
- `AST`: strong broad-audio spectrogram transformer, usually the best first serious candidate for non-speech audio tasks
- `WavLM`: speech self-supervised encoder, useful to test whether speech-style representations transfer to baby cries
- `Wav2Vec2`: lighter speech SSL baseline with broad ecosystem support

### Layer 2: optional fine-tuning

After the frozen-encoder bakeoff, pick one model and fine-tune it end-to-end. The notebook defaults to **AST first**, because baby cries are closer to general acoustic events than clean speech transcripts.

### Why not only one giant model?

Because Donate-a-Cry is tiny and highly imbalanced. On this dataset, transfer learning, calibration, and split hygiene usually matter more than chasing the biggest checkpoint.


## Research notes and why these models were chosen

The choices here are based on recent primary sources and official model docs:

- **AST**: a pure-attention spectrogram model that Hugging Face documents as achieving state-of-the-art audio classification results and recommends for downstream audio classification tasks. See the AST paper and docs:
  - https://huggingface.co/docs/transformers/en/model_doc/audio-spectrogram-transformer
- **WavLM**: Microsoft's large-scale self-supervised speech model, designed for broad speech tasks and documented as strong on speaker and paralinguistic tasks:
  - https://huggingface.co/docs/transformers/en/model_doc/wavlm
- **Wav2Vec2-BERT**: much larger and more recent multilingual audio representation model, pre-trained on 4.5M hours and explicitly documented as requiring fine-tuning for audio classification; this notebook keeps it as an optional heavy model because it is expensive for local runs:
  - https://huggingface.co/docs/transformers/en/model_doc/wav2vec2-bert
- **General Hugging Face audio classification workflow**:
  - https://huggingface.co/docs/transformers/en/tasks/audio_classification
- **Baby-cry transfer learning evidence**: a 2025 study on speech transformer representations for baby cries reports that latent representations from pre-trained speech models can effectively classify baby cries across multiple datasets, which supports a transfer-learning-first strategy here:
  - https://arxiv.org/abs/2509.02259

My judgment for **this** product is:

- start with **AST** as the main fine-tuning candidate,
- keep **WavLM** and **Wav2Vec2** as transfer-learning comparisons,
- treat very large checkpoints such as **Wav2Vec2-BERT** as optional heavy experiments,
- only move to more specialized or multimodal models after the data flywheel grows.


In [ ]:
# If needed, install notebook dependencies first.
# Restart the kernel after running this cell.
# %pip install -r ../requirements-notebook.txt


In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import sys
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from IPython.display import Audio, Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')


In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists() and (candidate / 'training').exists():
            return candidate
    raise FileNotFoundError('Could not find the repo root from the current working directory.')

PROJECT_ROOT = find_project_root()
TRAINING_ROOT = PROJECT_ROOT / 'training'
NOTEBOOK_ROOT = TRAINING_ROOT / 'notebooks'
OUTPUT_ROOT = TRAINING_ROOT / 'outputs' / 'notebook_bakeoff'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT = OUTPUT_ROOT / 'embedding_cache'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

if str(TRAINING_ROOT) not in sys.path:
    sys.path.insert(0, str(TRAINING_ROOT))

from train_baseline import (
    BabyCryCNN,
    MelDataset,
    build_class_weights,
    collect_records,
    evaluate,
    ensure_dataset,
    load_log_mel,
    pick_device,
    set_seed,
    split_records,
    train_one_epoch,
)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'TRAINING_ROOT = {TRAINING_ROOT}')
print(f'OUTPUT_ROOT = {OUTPUT_ROOT}')


In [ ]:
@dataclass
class ExperimentConfig:
    seed: int = 42
    sample_rate: int = 16_000
    duration_seconds: int = 7
    batch_size: int = 16
    cnn_epochs: int = 10
    cnn_learning_rate: float = 1e-3
    fine_tune_epochs: int = 8
    fine_tune_learning_rate: float = 2e-5
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15
    max_samples: int = 0  # set >0 for fast smoke tests
    device: str = str(pick_device())
    default_fine_tune_model: str = 'MIT/ast-finetuned-audioset-10-10-0.4593'

CONFIG = ExperimentConfig()
set_seed(CONFIG.seed)
print(CONFIG)


In [ ]:
DATA_ROOT = TRAINING_ROOT / 'data'
CLEANED_ROOT = ensure_dataset(DATA_ROOT)
ALL_RECORDS = collect_records(CLEANED_ROOT)

if CONFIG.max_samples > 0:
    rng = random.Random(CONFIG.seed)
    rng.shuffle(ALL_RECORDS)
    ALL_RECORDS = ALL_RECORDS[: CONFIG.max_samples]

TRAIN_RECORDS, VAL_RECORDS, TEST_RECORDS = split_records(
    ALL_RECORDS,
    train_ratio=CONFIG.train_ratio,
    val_ratio=CONFIG.val_ratio,
    test_ratio=CONFIG.test_ratio,
    seed=CONFIG.seed,
)

LABELS = sorted({record.app_label for record in ALL_RECORDS})
LABEL_TO_INDEX = {label: index for index, label in enumerate(LABELS)}
INDEX_TO_LABEL = {index: label for label, index in LABEL_TO_INDEX.items()}

print('Dataset root:', CLEANED_ROOT)
print('Label map:', LABEL_TO_INDEX)
print('Split sizes:', {
    'train': len(TRAIN_RECORDS),
    'val': len(VAL_RECORDS),
    'test': len(TEST_RECORDS),
})


In [ ]:
def records_to_frame(records):
    return pd.DataFrame([
        {
            'audio_path': record.audio_path,
            'app_label': record.app_label,
            'raw_label': record.raw_label,
            'group_id': record.group_id,
            'age_bucket': record.age_bucket,
            'gender': record.gender,
            'source_folder': record.source_folder,
        }
        for record in records
    ])

train_df = records_to_frame(TRAIN_RECORDS)
val_df = records_to_frame(VAL_RECORDS)
test_df = records_to_frame(TEST_RECORDS)
full_df = pd.concat([
    train_df.assign(split='train'),
    val_df.assign(split='val'),
    test_df.assign(split='test'),
], ignore_index=True)

summary = (
    full_df.groupby(['split', 'app_label'])
    .size()
    .rename('count')
    .reset_index()
    .sort_values(['split', 'app_label'])
)
summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.countplot(data=full_df, x='app_label', hue='split', ax=axes[0])
axes[0].set_title('Class distribution by split')
axes[0].tick_params(axis='x', rotation=20)

(
    full_df.groupby('app_label')['group_id']
    .nunique()
    .sort_values(ascending=False)
    .plot(kind='bar', ax=axes[1])
)
axes[1].set_title('Unique source groups per app label')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


In [ ]:
def load_audio_clip(audio_path: str | Path, sample_rate: int = CONFIG.sample_rate, duration_seconds: int = CONFIG.duration_seconds) -> np.ndarray:
    signal, _ = librosa.load(audio_path, sr=sample_rate, mono=True, duration=duration_seconds)
    target_len = sample_rate * duration_seconds
    if signal.shape[0] < target_len:
        signal = np.pad(signal, (0, target_len - signal.shape[0]))
    else:
        signal = signal[:target_len]
    return signal.astype(np.float32)

sample_row = full_df.sample(1, random_state=CONFIG.seed).iloc[0]
waveform = load_audio_clip(sample_row.audio_path)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(waveform, lw=0.8)
ax.set_title(f"Example waveform | label={sample_row.app_label} | split={sample_row.split}")
ax.set_xlabel('sample')
ax.set_ylabel('amplitude')
plt.show()

Audio(waveform, rate=CONFIG.sample_rate)


## 1. CNN spectrogram baseline

This is the same family as the existing baseline script, kept here as the **anchor model**. It is not the likely final production model, but it is useful for measuring whether a pretrained encoder really helps.


In [ ]:
def run_cnn_baseline(
    train_records=TRAIN_RECORDS,
    val_records=VAL_RECORDS,
    test_records=TEST_RECORDS,
    *,
    epochs: int = CONFIG.cnn_epochs,
    batch_size: int = CONFIG.batch_size,
    learning_rate: float = CONFIG.cnn_learning_rate,
):
    device = torch.device(CONFIG.device)
    model = BabyCryCNN(num_classes=len(LABELS)).to(device)
    train_loader = DataLoader(
        MelDataset(train_records, LABEL_TO_INDEX),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )
    val_loader = DataLoader(
        MelDataset(val_records, LABEL_TO_INDEX),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    test_loader = DataLoader(
        MelDataset(test_records, LABEL_TO_INDEX),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    class_weights = build_class_weights(train_records, LABEL_TO_INDEX).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    history = []
    best_state = None
    best_val_macro_f1 = -1.0

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = evaluate(model, val_loader, device)
        row = {'epoch': epoch, 'train_loss': train_loss, **val_metrics}
        history.append(row)
        print(row)
        if val_metrics['macro_f1'] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics['macro_f1']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    test_metrics = evaluate(model, test_loader, device)
    history_df = pd.DataFrame(history)
    return {
        'model_name': 'mel_cnn',
        'best_val_macro_f1': best_val_macro_f1,
        'test_metrics': test_metrics,
        'history': history_df,
        'state_dict': best_state,
    }


In [ ]:
RUN_CNN_BASELINE = True

cnn_result = None
if RUN_CNN_BASELINE:
    cnn_result = run_cnn_baseline()
    display(cnn_result['history'])
    display(pd.DataFrame([{'model': 'mel_cnn', **cnn_result['test_metrics']}]))


## 2. Frozen pretrained encoders + linear probe

This is the fastest way to compare multiple modern audio encoders on a small, noisy dataset.

The logic is:

1. load a pretrained encoder,
2. extract one embedding per clip,
3. fit a lightweight classifier on the training embeddings,
4. rank models by validation and test macro F1.

This is usually the highest-signal first experiment when the downstream dataset is small.


In [ ]:
from transformers import AutoFeatureExtractor, AutoModel

EMBEDDING_MODELS = {
    'ast_audioset': {
        'hf_id': 'MIT/ast-finetuned-audioset-10-10-0.4593',
        'family': 'general_audio',
        'enabled': True,
        'note': 'Best first serious candidate for baby cries.',
    },
    'wavlm_base_plus': {
        'hf_id': 'microsoft/wavlm-base-plus',
        'family': 'speech_ssl',
        'enabled': True,
        'note': 'Strong speech/paralinguistic transfer baseline.',
    },
    'wav2vec2_base': {
        'hf_id': 'facebook/wav2vec2-base',
        'family': 'speech_ssl',
        'enabled': True,
        'note': 'Lighter speech SSL baseline with wide ecosystem support.',
    },
    'w2vbert_large_optional': {
        'hf_id': 'facebook/wav2vec2-bert-rel-pos-large',
        'family': 'heavy_multilingual_ssl',
        'enabled': False,
        'note': 'Optional heavy model; enable only if you have the RAM and time.',
    },
}

pd.DataFrame(EMBEDDING_MODELS).T


In [ ]:
def mean_pool_hidden_states(model_outputs) -> torch.Tensor:
    if hasattr(model_outputs, 'pooler_output') and model_outputs.pooler_output is not None:
        return model_outputs.pooler_output
    hidden = getattr(model_outputs, 'last_hidden_state', None)
    if hidden is None:
        if isinstance(model_outputs, tuple) and model_outputs:
            hidden = model_outputs[0]
        else:
            raise ValueError('Could not find hidden states in model outputs.')
    if hidden.ndim == 2:
        return hidden
    return hidden.mean(dim=1)


def cache_key(model_key: str, split_name: str) -> Path:
    return CACHE_ROOT / f'{model_key}_{split_name}.npz'


def extract_embeddings(records, model_key: str, model_cfg: dict[str, Any], split_name: str, batch_size: int = 4):
    cache_path = cache_key(model_key, split_name)
    if cache_path.exists():
        cached = np.load(cache_path)
        return cached['embeddings'], cached['labels']

    feature_extractor = AutoFeatureExtractor.from_pretrained(model_cfg['hf_id'])
    model = AutoModel.from_pretrained(model_cfg['hf_id'])
    device = torch.device(CONFIG.device)
    model = model.to(device)
    model.eval()

    all_embeddings = []
    all_labels = []

    for start in tqdm(range(0, len(records), batch_size), desc=f'extract:{model_key}:{split_name}'):
        batch_records = records[start:start + batch_size]
        audio_batch = [load_audio_clip(record.audio_path) for record in batch_records]
        inputs = feature_extractor(
            audio_batch,
            sampling_rate=CONFIG.sample_rate,
            padding=True,
            return_tensors='pt',
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.inference_mode():
            outputs = model(**inputs)
            embeddings = mean_pool_hidden_states(outputs).detach().cpu().numpy()
        labels = np.array([LABEL_TO_INDEX[record.app_label] for record in batch_records], dtype=np.int64)
        all_embeddings.append(embeddings)
        all_labels.append(labels)

    X = np.concatenate(all_embeddings, axis=0)
    y = np.concatenate(all_labels, axis=0)
    np.savez(cache_path, embeddings=X, labels=y)
    return X, y


def compute_multiclass_metrics(y_true, probabilities):
    predictions = probabilities.argmax(axis=1)
    top2 = np.argsort(probabilities, axis=1)[:, -2:]
    top2_hits = [int(target in row) for target, row in zip(y_true, top2)]
    return {
        'accuracy': float(accuracy_score(y_true, predictions)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, predictions)),
        'macro_f1': float(f1_score(y_true, predictions, average='macro')),
        'top2_accuracy': float(np.mean(top2_hits)),
    }


def run_linear_probe(model_key: str, model_cfg: dict[str, Any]):
    X_train, y_train = extract_embeddings(TRAIN_RECORDS, model_key, model_cfg, 'train')
    X_val, y_val = extract_embeddings(VAL_RECORDS, model_key, model_cfg, 'val')
    X_test, y_test = extract_embeddings(TEST_RECORDS, model_key, model_cfg, 'test')

    classifier = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', multi_class='auto')),
    ])
    classifier.fit(X_train, y_train)

    val_prob = classifier.predict_proba(X_val)
    test_prob = classifier.predict_proba(X_test)

    return {
        'model': model_key,
        'hf_id': model_cfg['hf_id'],
        'family': model_cfg['family'],
        'val': compute_multiclass_metrics(y_val, val_prob),
        'test': compute_multiclass_metrics(y_test, test_prob),
        'classifier': classifier,
    }


In [ ]:
DEFAULT_MODEL_KEYS = [key for key, cfg in EMBEDDING_MODELS.items() if cfg['enabled']]
DEFAULT_MODEL_KEYS


In [ ]:
probe_results = []
for model_key in DEFAULT_MODEL_KEYS:
    result = run_linear_probe(model_key, EMBEDDING_MODELS[model_key])
    probe_results.append(result)

probe_table = pd.DataFrame([
    {
        'model': result['model'],
        'family': result['family'],
        'hf_id': result['hf_id'],
        'val_macro_f1': result['val']['macro_f1'],
        'val_top2': result['val']['top2_accuracy'],
        'test_macro_f1': result['test']['macro_f1'],
        'test_top2': result['test']['top2_accuracy'],
        'test_accuracy': result['test']['accuracy'],
        'test_balanced_accuracy': result['test']['balanced_accuracy'],
    }
    for result in probe_results
]).sort_values(['val_macro_f1', 'test_macro_f1'], ascending=False)

probe_table


In [ ]:
if not probe_table.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.barplot(data=probe_table, x='model', y='val_macro_f1', ax=axes[0])
    axes[0].set_title('Validation macro F1')
    axes[0].tick_params(axis='x', rotation=20)

    sns.barplot(data=probe_table, x='model', y='test_top2', ax=axes[1])
    axes[1].set_title('Test top-2 accuracy')
    axes[1].tick_params(axis='x', rotation=20)
    plt.tight_layout()
    plt.show()


### Interpreting the bakeoff

For this dataset, **macro F1** matters more than raw accuracy because `Hungry` dominates the label distribution.

A good outcome usually looks like this:

- AST or another broad-audio model wins on macro F1,
- speech SSL models remain competitive on top-2 accuracy,
- the CNN baseline trails the best transfer models unless heavily tuned.

If the frozen-encoder probe already gives a strong result, that is a strong signal that the representation is useful enough to fine-tune.


## 3. Pick a fine-tuning candidate

The default recommendation is to fine-tune **AST first**. That choice is not ideological; it is because baby cries are non-linguistic sounds, and AST is built for broad audio classification rather than speech recognition.

If the frozen bakeoff proves WavLM or Wav2Vec2 is stronger on your machine and split, change `FINE_TUNE_MODEL_NAME` below.


In [ ]:
FINE_TUNE_MODEL_NAME = CONFIG.default_fine_tune_model
FINE_TUNE_BATCH_SIZE = 4
RUN_FINE_TUNING = False

print('Fine-tune target:', FINE_TUNE_MODEL_NAME)


In [ ]:
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    Trainer,
    TrainingArguments,
)

class HFAudioDataset(Dataset):
    def __init__(self, records, feature_extractor, label_to_index, sample_rate=CONFIG.sample_rate):
        self.records = records
        self.feature_extractor = feature_extractor
        self.label_to_index = label_to_index
        self.sample_rate = sample_rate

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        waveform = load_audio_clip(record.audio_path, sample_rate=self.sample_rate)
        encoded = self.feature_extractor(
            waveform,
            sampling_rate=self.sample_rate,
            return_tensors='pt',
            padding=False,
        )
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item['labels'] = torch.tensor(self.label_to_index[record.app_label], dtype=torch.long)
        return item


class AudioDataCollator:
    def __init__(self, feature_extractor):
        self.feature_extractor = feature_extractor

    def __call__(self, features):
        labels = torch.tensor([feature['labels'] for feature in features], dtype=torch.long)
        feature_dicts = [{k: v for k, v in feature.items() if k != 'labels'} for feature in features]
        batch = self.feature_extractor.pad(feature_dicts, return_tensors='pt')
        batch['labels'] = labels
        return batch


class WeightedAudioTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device) if self.class_weights is not None else None)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def compute_trainer_metrics(eval_pred):
    logits, labels = eval_pred
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    return compute_multiclass_metrics(labels, probabilities)


In [ ]:
def run_fine_tuning(model_name: str = FINE_TUNE_MODEL_NAME):
    feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)

    train_dataset = HFAudioDataset(TRAIN_RECORDS, feature_extractor, LABEL_TO_INDEX)
    val_dataset = HFAudioDataset(VAL_RECORDS, feature_extractor, LABEL_TO_INDEX)
    test_dataset = HFAudioDataset(TEST_RECORDS, feature_extractor, LABEL_TO_INDEX)

    model = AutoModelForAudioClassification.from_pretrained(
        model_name,
        num_labels=len(LABELS),
        label2id=LABEL_TO_INDEX,
        id2label=INDEX_TO_LABEL,
        ignore_mismatched_sizes=True,
    )

    class_weights = build_class_weights(TRAIN_RECORDS, LABEL_TO_INDEX)
    run_dir = OUTPUT_ROOT / f"finetune_{model_name.replace('/', '_')}"

    args = TrainingArguments(
        output_dir=str(run_dir),
        evaluation_strategy='epoch',
        save_strategy='epoch',
        logging_strategy='steps',
        logging_steps=10,
        learning_rate=CONFIG.fine_tune_learning_rate,
        per_device_train_batch_size=FINE_TUNE_BATCH_SIZE,
        per_device_eval_batch_size=FINE_TUNE_BATCH_SIZE,
        num_train_epochs=CONFIG.fine_tune_epochs,
        warmup_ratio=0.1,
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        report_to='none',
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        remove_unused_columns=False,
    )

    trainer = WeightedAudioTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=AudioDataCollator(feature_extractor),
        compute_metrics=compute_trainer_metrics,
        class_weights=class_weights,
    )

    trainer.train()
    val_metrics = trainer.evaluate(val_dataset)
    test_metrics = trainer.evaluate(test_dataset, metric_key_prefix='test')

    (run_dir / 'final_metrics.json').write_text(
        json.dumps({'val': val_metrics, 'test': test_metrics}, indent=2),
        encoding='utf-8',
    )
    return trainer, val_metrics, test_metrics


In [ ]:
trainer = None
fine_tune_val_metrics = None
fine_tune_test_metrics = None

if RUN_FINE_TUNING:
    trainer, fine_tune_val_metrics, fine_tune_test_metrics = run_fine_tuning(FINE_TUNE_MODEL_NAME)
    display(pd.DataFrame([
        {'split': 'val', **fine_tune_val_metrics},
        {'split': 'test', **fine_tune_test_metrics},
    ]))
else:
    print('Fine-tuning is disabled. Set RUN_FINE_TUNING = True when you are ready.')


## 4. Recommended decision rule

Use the notebook like this:

1. Run the **CNN baseline** once as an anchor.
2. Run the **frozen encoder bakeoff** and rank models by validation macro F1.
3. Fine-tune the best one or two candidates.
4. Choose the winner using:
   - validation macro F1,
   - test macro F1,
   - top-2 accuracy,
   - training stability,
   - inference cost.

For the current Donate-a-Cry setup, my default recommendation is:

- **fine-tune AST first**,
- keep **WavLM** as the strongest alternative,
- treat **Wav2Vec2-BERT** as an optional later experiment, not the first local baseline.


## 5. Suggested next experiments

After this notebook is stable, the highest-value next steps are:

- add **cross-validation by group id** instead of a single split,
- save **prediction probabilities** and calibration plots,
- add an **uncertain / abstain** threshold,
- compare **data augmentation** such as time masking, additive room noise, and gain jitter,
- evaluate whether **uploaded video audio tracks** behave differently from direct recordings,
- later: add lightweight visual cues for late fusion, not end-to-end video training first.
